# G12: G9's eval_mini on OLMoE GGUF files (Colab T4)

Runs G9's **mini** profile (ARC-C 500, HellaSwag 400, MMLU 570, GSM8K 200, TruthfulQA-mc2 200; G9's shots, limits
and seed) on **bf16 and six llama.cpp formats**, all through one backend (llama-cpp-python in-process; see
`llamacpp_lm.py` for why not llama-server), with **per-item logs**, then compares each format with bf16 item by
item (`paired.py`).

**Runtime:** T4 GPU. Then **Run all**. Every step stops the run if it fails.

- Output goes to Google Drive when it can be mounted, else to `/content/eval_mini_out` (download the zip at the
  end). Each task is saved as soon as it finishes; **Run all again after a disconnect** and finished tasks are
  skipped. The compiled llama-cpp-python is cached next to the output, so a second session skips the compile.
- First run: keep `LIMIT = 5` (~30-40 min, mostly the one-time compile and model conversion). Then set
  `LIMIT = None` for the full run (estimate: ~25 min per quantized format, ~1-1.5 h for bf16).

In [ ]:
# --- settings -----------------------------------------------------------------------------------------------
LIMIT     = 5          # items per task for a test run (MMLU: per subject); None = G9's full mini profile
FORMATS   = ["bf16", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "Q2_K"]   # bf16 first: it is the baseline
BASE      = "bf16"
MODEL, REVISION = "allenai/OLMoE-1B-7B-0924", "6d84c48581ece794365f2b8e9cfb043c68ade9c5"  # team pin
LLAMA_TAG = "b11669"           # llama.cpp build of the team's conversion/quantization
LLAMA_CPP_PYTHON = "0.3.36"    # in-process log-probabilities checked correct with this version
G9_RAW = "https://raw.githubusercontent.com/theschoolofai/era-v5-lab/main/teams/09-evaluation/harness/run_eval_mini.py"
DRIVE_DIR = "/content/drive/MyDrive/era-v5-g12/eval_mini"   # used when Drive mounts
LOCAL_OUT = "/content/eval_mini_out"                        # used otherwise
WORK      = "/content/work"                                 # model files, rebuilt per session
BF16_FALLBACK_GPU_LAYERS = 12  # of 16 layers, only if bf16 does not fit on the GPU

import os
if os.environ.get("EVAL_MINI_LOCAL_TEST"):   # dry run of these same cells on a Mac (see build_notebook.py)
    LIMIT, FORMATS, BASE = 2, ["Q8_0", "Q2_K"], "Q8_0"
    LOCAL_OUT = os.environ.get("EVAL_MINI_OUT", "/tmp/eval_mini_local")
    WORK = os.path.expanduser("~/.cache/era-v5-quant")

In [ ]:
import datetime, glob, json, os, platform, shutil, subprocess, sys, time
LOCAL_TEST = bool(os.environ.get("EVAL_MINI_LOCAL_TEST"))
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUT = LOCAL_OUT
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = DRIVE_DIR
    except Exception as e:  # e.g. not available from an editor-attached runtime
        print(f"Drive not mounted ({e!r}); output goes to {LOCAL_OUT}. DOWNLOAD THE ZIP AT THE END.")
for d in (OUT, f"{OUT}/logs", WORK):
    os.makedirs(d, exist_ok=True)
CODE = f"{WORK}/eval_mini_code"; os.makedirs(CODE, exist_ok=True)

def sh(cmd, log=None):
    """Run a shell command; stream its output (and append it to a log); stop the notebook if it fails."""
    print("$", cmd, flush=True)
    full = f"set -o pipefail; ( {cmd} ) 2>&1" + (f" | tee -a {log}" if log else "")
    rc = subprocess.run(full, shell=True, executable="/bin/bash").returncode
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}" + (f"; see {log}" if log else ""))

ENV = {"started": datetime.datetime.now().isoformat(timespec="seconds"), "in_colab": IN_COLAB, "local_test": LOCAL_TEST,
       "output_dir": OUT, "settings": {k: globals()[k] for k in ("LIMIT", "FORMATS", "BASE", "MODEL", "REVISION",
       "LLAMA_TAG", "LLAMA_CPP_PYTHON", "BF16_FALLBACK_GPU_LAYERS")}, "python": platform.python_version(), "formats": {}}
def save_env():
    json.dump(ENV, open(f"{OUT}/environment.json", "w"), indent=1)
if shutil.which("nvidia-smi"):
    ENV["gpu"] = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                                shell=True, capture_output=True, text=True).stdout.strip()
    sh("nvidia-smi")
sh(f"df -h {WORK} | tail -1")
save_env()
print("output:", OUT)

In [ ]:
# the evaluation code (llamacpp_lm.py, run_mini_gguf.py, paired.py), embedded by build_notebook.py
SOURCES = {
"llamacpp_lm.py": "\"\"\"lm-evaluation-harness model class that runs a GGUF file in-process with llama-cpp-python.\n\nWhy this exists: lm_eval scores multiple-choice tasks by the log-probability of each answer. Neither server\nroute gives correct numbers for GGUF files (checked 28 Oct, llama.cpp b11669, llama-cpp-python 0.3.36):\n  - llama-server ignores `echo=True` and returns logprobs only for generated tokens, so lm_eval's\n    local-completions / gguf backends cannot score multiple choice at all;\n  - llama-cpp-python's server returns prompt logprobs SHIFTED BY ONE position (log p(\" Paris\" | \"The capital\n    of France is\") comes back as -8.58 instead of -0.49), which lm_eval would sum without any error.\nHere the model runs in-process with every position's logits kept, and the log-probability of token i is read\nfrom the distribution at position i-1. Checked: log p(\" Paris\" | ...) = -0.49, as greedy generation reports.\n\nUse: `from llamacpp_lm import LlamaCppLM; lm = LlamaCppLM(\"model.gguf\")`, then pass `model=lm` to\n`lm_eval.simple_evaluate`. Requests are scored one at a time; a cache of the previous sequence's tokens means\nthe shared question of a multiple-choice item is evaluated once, not once per answer option.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nfrom llama_cpp import Llama\nfrom lm_eval.api.model import LM\nfrom tqdm import tqdm\n\n\nclass LlamaCppLM(LM):\n    def __init__(self, model_path: str, n_ctx: int = 2048, n_gpu_layers: int = -1, max_gen_toks: int = 256,\n                 seed: int = 42, n_batch: int = 512):\n        super().__init__()\n        self.llm = Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, logits_all=True,\n                         seed=seed, n_batch=n_batch, verbose=False)\n        self.n_ctx, self.max_gen_toks = n_ctx, max_gen_toks\n        meta = self.llm.metadata\n        self.add_bos = meta.get(\"tokenizer.ggml.add_bos_token\", \"false\") == \"true\"\n        bos, eos = self.llm.token_bos(), self.llm.token_eos()\n        # lm_eval's HF model uses this token as the context of an empty-context request\n        self.prefix_token = bos if self.add_bos and bos >= 0 else eos\n\n    # ------------------------------------------------------------------ helpers\n    def _tok(self, text: str, bos: bool) -> list[int]:\n        return self.llm.tokenize(text.encode(\"utf-8\"), add_bos=bos, special=False)\n\n    def _encode_pair(self, context: str, continuation: str) -> tuple[list[int], list[int]]:\n        \"\"\"Same split as lm_eval's HFLM: trailing spaces of the context move to the continuation, then the\n        continuation tokens are the tail of the jointly tokenized string (so merges across the boundary hold).\"\"\"\n        n_spaces = len(context) - len(context.rstrip())\n        if n_spaces:\n            continuation, context = context[-n_spaces:] + continuation, context[:-n_spaces]\n        if not context:\n            return [self.prefix_token], self._tok(continuation, bos=False)\n        whole = self._tok(context + continuation, bos=self.add_bos)\n        ctx = self._tok(context, bos=self.add_bos)\n        return ctx, whole[len(ctx):]\n\n    def _eval_reuse_prefix(self, tokens: list[int]) -> None:\n        \"\"\"Evaluate `tokens`, keeping the KV cache and logit rows of the longest prefix shared with the previous\n        sequence. Row j of llm.scores depends only on tokens[:j+1], so kept rows stay valid.\"\"\"\n        llm, keep = self.llm, 0\n        for a, b in zip(llm.input_ids[: llm.n_tokens], tokens):\n            if a != b:\n                break\n            keep += 1\n        keep = min(keep, len(tokens) - 1)  # always evaluate at least the last token\n        llm.n_tokens = keep                # eval() drops the KV cache from n_tokens onward\n        llm.eval(tokens[keep:])\n\n    def _score(self, context_tokens: list[int], continuation_tokens: list[int]) -> tuple[float, bool]:\n        tokens = (context_tokens + continuation_tokens)[-self.n_ctx:]   # left-truncate, as HFLM does\n        n_cont = len(continuation_tokens)\n        self._eval_reuse_prefix(tokens)\n        start = len(tokens) - n_cont\n        rows = np.asarray(self.llm.scores[start - 1: len(tokens) - 1], dtype=np.float64)  # row i-1 predicts token i\n        logsm = rows - np.logaddexp.reduce(rows, axis=1, keepdims=True)\n        targets = np.asarray(tokens[start:])\n        logprob = float(logsm[np.arange(n_cont), targets].sum())\n        greedy = bool((logsm.argmax(axis=1) == targets).all())\n        return logprob, greedy\n\n    # ------------------------------------------------------------------ lm_eval interface\n    def loglikelihood(self, requests, disable_tqdm: bool = False):\n        pairs = [self._encode_pair(*req.args) for req in requests]\n        # score requests that share a question back to back, so the prefix cache is used; return in input order\n        order = sorted(range(len(pairs)), key=lambda i: pairs[i][0])\n        out = [None] * len(pairs)\n        for i in tqdm(order, disable=disable_tqdm, desc=\"loglikelihood\"):\n            out[i] = self._score(*pairs[i])\n        return out\n\n    def loglikelihood_rolling(self, requests, disable_tqdm: bool = False):\n        raise NotImplementedError(\"not needed by the eval_mini profiles (perplexity is measured in perplexity/)\")\n\n    def generate_until(self, requests, disable_tqdm: bool = False):\n        out = []\n        for req in tqdm(requests, disable=disable_tqdm, desc=\"generate_until\"):\n            context, gen_kwargs = req.args\n            gen_kwargs = dict(gen_kwargs)\n            until = gen_kwargs.pop(\"until\", None) or []\n            until = [until] if isinstance(until, str) else list(until)\n            max_new = int(gen_kwargs.pop(\"max_gen_toks\", self.max_gen_toks))\n            if gen_kwargs.get(\"do_sample\") or float(gen_kwargs.get(\"temperature\", 0.0) or 0.0) > 0:\n                raise ValueError(f\"only greedy generation is supported, got {gen_kwargs}\")\n            prompt = self._tok(context, bos=self.add_bos)[-(self.n_ctx - max_new):]\n            self.llm.reset()  # generation manages its own cache; do not mix it with the scoring cache\n            res = self.llm.create_completion(prompt=prompt, max_tokens=max_new, temperature=0.0, top_k=1,\n                                             stop=until or None)\n            out.append(res[\"choices\"][0][\"text\"])\n        self.llm.reset()\n        return out\n",
"run_mini_gguf.py": "#!/usr/bin/env python3\n\"\"\"Run G9's eval_mini profile on one GGUF file, with per-item logs, in G9's eval_mini_v0 record format.\n\n    python run_mini_gguf.py --gguf OLMoE-1B-7B-0924-Q8_0.gguf --format Q8_0 --out runs/ \\\n        --g9-harness <era-v5-lab main>/teams/09-evaluation/harness/run_eval_mini.py\n\nThe task set, shot counts, limits, seed and record format are imported from G9's run_eval_mini.py, not copied,\nso a run here is like for like with G9's baseline. Per task it writes:\n  <out>/<format>/<task>_<k>shot.samples.jsonl   one line per item: task, doc_id, target, response, metric values\n  <out>/<format>/summary.json                   run metadata + one record per (task, metric, filter)\nA task whose records are already in summary.json is skipped, so an interrupted run resumes where it stopped.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, datetime, hashlib, importlib.util, json, platform, subprocess, sys, time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE))\n\n\ndef load_g9(path: Path):\n    spec = importlib.util.spec_from_file_location(\"run_eval_mini\", path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[\"run_eval_mini\"] = mod   # its dataclasses look themselves up in sys.modules while being defined\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef gpu_name() -> str:\n    try:\n        return subprocess.run([\"nvidia-smi\", \"--query-gpu=name,memory.total\", \"--format=csv,noheader\"],\n                              capture_output=True, text=True, check=True).stdout.strip().replace(\"\\n\", \" + \")\n    except (OSError, subprocess.CalledProcessError):\n        if platform.system() == \"Darwin\":\n            sc = lambda k: subprocess.run([\"sysctl\", \"-n\", k], capture_output=True, text=True).stdout.strip()\n            return f\"{sc('machdep.cpu.brand_string')} {int(sc('hw.memsize')) // 2**30}GB (Metal)\"\n        return platform.processor() or platform.machine()\n\n\ndef sample_rows(results: dict, metric_names: set[str]) -> list[dict]:\n    \"\"\"Flatten lm_eval's per-sample logs (one list per leaf task) into slim rows keyed by (task, filter, doc_id);\n    GSM8K logs each item once per answer filter (strict-match, flexible-extract).\"\"\"\n    rows = []\n    for task, samples in (results.get(\"samples\") or {}).items():\n        for s in samples:\n            resp = s.get(\"filtered_resps\")\n            rows.append({\"task\": task, \"doc_id\": s[\"doc_id\"], \"filter\": s.get(\"filter\", \"none\"), \"target\": s.get(\"target\"),\n                         \"response\": resp if not isinstance(resp, list) or len(resp) > 1 else resp[0],\n                         **{m: s[m] for m in metric_names if m in s}})\n    return rows\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--gguf\", required=True, type=Path)\n    ap.add_argument(\"--format\", required=True, help=\"label for this file, e.g. bf16, Q8_0\")\n    ap.add_argument(\"--out\", required=True, type=Path)\n    ap.add_argument(\"--g9-harness\", required=True, type=Path, help=\"path to G9's run_eval_mini.py\")\n    ap.add_argument(\"--profile\", default=\"mini\")\n    ap.add_argument(\"--tasks\", nargs=\"+\", help=\"subset of the profile's tasks\")\n    ap.add_argument(\"--limit\", type=int, help=\"override every task's limit (smoke tests only)\")\n    ap.add_argument(\"--n-ctx\", type=int, default=2048)\n    ap.add_argument(\"--n-gpu-layers\", type=int, default=-1, help=\"-1 = all layers on the GPU\")\n    ap.add_argument(\"--model\", default=\"allenai/OLMoE-1B-7B-0924\")\n    ap.add_argument(\"--revision\", default=\"6d84c48581ece794365f2b8e9cfb043c68ade9c5\")\n    args = ap.parse_args()\n\n    g9 = load_g9(args.g9_harness)\n    profile = g9.PROFILES[args.profile]\n    tasks = args.tasks or list(profile)\n    out = args.out / args.format; out.mkdir(parents=True, exist_ok=True)\n    summary_path = out / \"summary.json\"\n    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {\n        \"schema_version\": g9.SCHEMA_VERSION, \"run_meta\": {}, \"records\": []}\n    done = {r[\"task\"] for r in summary[\"records\"]}\n\n    import llama_cpp, lm_eval\n    from llamacpp_lm import LlamaCppLM\n    t0 = time.perf_counter()\n    lm = LlamaCppLM(str(args.gguf), n_ctx=args.n_ctx, n_gpu_layers=args.n_gpu_layers, seed=g9.DEFAULT_SEED)\n    load_s = time.perf_counter() - t0\n    summary[\"run_meta\"] = {\n        \"model\": args.model, \"model_revision\": args.revision, \"format\": args.format,\n        \"gguf\": args.gguf.name, \"gguf_bytes\": args.gguf.stat().st_size, \"profile\": args.profile,\n        \"backend\": f\"llama-cpp-python {llama_cpp.__version__} (in-process, logits_all)\",\n        \"lm_eval\": lm_eval.__version__, \"hardware\": gpu_name(), \"n_ctx\": args.n_ctx,\n        \"n_gpu_layers\": args.n_gpu_layers, \"seed\": g9.DEFAULT_SEED, \"load_s\": round(load_s, 1),\n        \"g9_harness_sha256\": hashlib.sha256(args.g9_harness.read_bytes()).hexdigest()[:16],\n    }\n    for name in tasks:\n        if name in done:\n            print(f\"{args.format}/{name}: already done, skipped\", flush=True); continue\n        spec, run = g9.TASKS[name], profile[name]\n        limit = args.limit if args.limit is not None else run.limit\n        t = time.perf_counter()\n        results = lm_eval.simple_evaluate(\n            model=lm, tasks=[name], num_fewshot=run.num_fewshot or None, limit=limit, log_samples=True,\n            random_seed=g9.DEFAULT_SEED, numpy_random_seed=g9.DEFAULT_SEED, torch_random_seed=g9.DEFAULT_SEED,\n            fewshot_random_seed=g9.DEFAULT_SEED)\n        secs = time.perf_counter() - t\n        context = {\"model\": args.model, \"model_revision\": args.revision, \"dtype\": args.format,\n                   \"profile\": args.profile, \"limit\": limit, \"seed\": g9.DEFAULT_SEED,\n                   \"harness_version\": lm_eval.__version__, \"run_date\": datetime.date.today().isoformat(),\n                   \"device\": \"llama-cpp-python\", \"hardware\": summary[\"run_meta\"][\"hardware\"], \"batch_size\": 1,\n                   \"wall_clock_s\": round(secs, 1)}\n        records = g9.extract_records(results, spec, run.num_fewshot, context)\n        metric_names = {r[\"metric\"] for r in records}\n        rows = sample_rows(results, metric_names)\n        with open(out / f\"{name}_{run.num_fewshot}shot.samples.jsonl\", \"w\") as fh:\n            for row in rows:\n                fh.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n        summary[\"records\"] = [r for r in summary[\"records\"] if r[\"task\"] != name] + records\n        summary_path.write_text(json.dumps(summary, indent=1))   # after every task: a crash loses one task at most\n        prim = next((r for r in records if r[\"primary\"]), records[0])\n        print(f\"{args.format}/{name}: {prim['metric']}={prim['value']:.4f} +/- {prim['stderr'] or 0:.4f} \"\n              f\"n={prim['n_items']} in {secs:.0f}s ({secs / max(prim['n_items'], 1):.2f}s/item), \"\n              f\"{len(rows)} per-item rows\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"paired.py": "#!/usr/bin/env python3\n\"\"\"Paired comparison of each quantized format with bf16 on eval_mini, item by item.\n\n    python paired.py runs/ --base bf16 [--out paired.md]\n\nReads runs/<format>/<task>_<k>shot.samples.jsonl (written by run_mini_gguf.py) and compares every format with\nthe base on the SAME items. Comparing two accuracies with their own standard errors (as compare_v0.md does for\nmodels) needs gaps of ~2 points per 400 items; pairing removes the item-to-item variation both runs share, so a\nmuch smaller real change becomes measurable.\n\nPer task, on its primary metric (G9's choice: acc_norm for ARC-C and HellaSwag, acc for MMLU and TruthfulQA-mc2,\nexact_match with the flexible-extract filter for GSM8K):\n  0/1 metrics:      b = items right in base but wrong in the format, c = the reverse; difference = (c - b) / n,\n                    95% CI from the paired variance; p from McNemar's exact test on b and c.\n  TruthfulQA-mc2:   acc is a probability share per item; mean of the per-item differences, 95% CI, no p-value.\nMMLU's 57 subjects are pooled into one row.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, json, math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nPRIMARY = {\"arc_challenge\": (\"acc_norm\", \"none\"), \"arc_easy\": (\"acc_norm\", \"none\"), \"hellaswag\": (\"acc_norm\", \"none\"),\n           \"mmlu\": (\"acc\", \"none\"), \"gsm8k\": (\"exact_match\", \"flexible-extract\"), \"truthfulqa_mc2\": (\"acc\", \"none\")}\nBINARY = {\"arc_challenge\", \"arc_easy\", \"hellaswag\", \"mmlu\", \"gsm8k\"}\n\n\ndef group(task: str) -> str:\n    return \"mmlu\" if task.startswith(\"mmlu\") else task\n\n\ndef load(fmt_dir: Path) -> dict[str, dict[tuple, float]]:\n    \"\"\"{group task: {(leaf task, doc_id): primary metric value}}\"\"\"\n    out = defaultdict(dict)\n    for f in sorted(fmt_dir.glob(\"*.samples.jsonl\")):\n        for line in open(f):\n            r = json.loads(line)\n            g = group(r[\"task\"])\n            if g not in PRIMARY:\n                continue\n            metric, filt = PRIMARY[g]\n            if r.get(\"filter\", \"none\") == filt and metric in r:\n                out[g][(r[\"task\"], r[\"doc_id\"])] = float(r[metric])\n    return out\n\n\ndef mcnemar_p(b: int, c: int) -> float:\n    n = b + c\n    if n == 0:\n        return 1.0\n    k = min(b, c)\n    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)\n\n\ndef compare(base: dict[tuple, float], other: dict[tuple, float], binary: bool) -> dict:\n    keys = sorted(base.keys() & other.keys())\n    n = len(keys)\n    if n == 0:\n        return {\"n\": 0}\n    x, y = [base[k] for k in keys], [other[k] for k in keys]\n    res = {\"n\": n, \"base\": sum(x) / n, \"fmt\": sum(y) / n, \"missing\": len(base.keys() ^ other.keys())}\n    if binary:\n        b = sum(1 for p, q in zip(x, y) if p == 1 and q == 0)\n        c = sum(1 for p, q in zip(x, y) if p == 0 and q == 1)\n        diff = (c - b) / n\n        se = math.sqrt(max((b + c) - (c - b) ** 2 / n, 0)) / n\n        res.update(diff=diff, lo=diff - 1.96 * se, hi=diff + 1.96 * se, b=b, c=c, p=mcnemar_p(b, c))\n    else:\n        d = [q - p for p, q in zip(x, y)]\n        mean = sum(d) / n\n        sd = math.sqrt(sum((v - mean) ** 2 for v in d) / (n - 1)) if n > 1 else 0.0\n        res.update(diff=mean, lo=mean - 1.96 * sd / math.sqrt(n), hi=mean + 1.96 * sd / math.sqrt(n))\n    return res\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"runs\", type=Path)\n    ap.add_argument(\"--base\", default=\"bf16\")\n    ap.add_argument(\"--out\", type=Path, help=\"also write the markdown table here\")\n    args = ap.parse_args()\n\n    base = load(args.runs / args.base)\n    lines = [f\"Paired against **{args.base}** on the same items. Differences in points (format minus base); \"\n             \"b = right in base, wrong in format; c = the reverse.\", \"\",\n             \"| Format | Task | n | Base | Format | Difference (95% CI) | b / c | McNemar p |\",\n             \"|---|---|---|---|---|---|---|---|\"]\n    for fmt_dir in sorted(p for p in args.runs.iterdir() if p.is_dir() and p.name != args.base):\n        other = load(fmt_dir)\n        for task in PRIMARY:\n            if task not in base or task not in other:\n                continue\n            r = compare(base[task], other[task], task in BINARY)\n            if not r[\"n\"]:\n                continue\n            pts = lambda v: f\"{100 * v:+.1f}\"\n            bc = f\"{r['b']} / {r['c']}\" if \"b\" in r else \"-\"\n            p = f\"{r['p']:.3f}\" if \"p\" in r else \"-\"\n            warn = f\" ({r['missing']} items unmatched)\" if r[\"missing\"] else \"\"\n            lines.append(f\"| {fmt_dir.name} | {task}{warn} | {r['n']} | {100 * r['base']:.1f} | {100 * r['fmt']:.1f} | \"\n                         f\"{pts(r['diff'])} ({pts(r['lo'])} to {pts(r['hi'])}) | {bc} | {p} |\")\n    text = \"\\n\".join(lines)\n    print(text)\n    if args.out:\n        args.out.write_text(text + \"\\n\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
for _name, _text in SOURCES.items():
    open(f"{CODE}/{_name}", "w").write(_text)
print("wrote", sorted(SOURCES), "to", CODE)

In [ ]:
# llama-cpp-python with CUDA, lm_eval and datasets at G9's pins. The CUDA build is compiled once (~15-25 min on
# Colab's 2 CPUs) into a wheel kept in OUT/wheels; later sessions install it in seconds.
if not LOCAL_TEST:
    wheels = glob.glob(f"{OUT}/wheels/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
    if not wheels:
        os.makedirs(f"{OUT}/wheels", exist_ok=True)
        sh(f'CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" pip wheel --no-deps --no-cache-dir '
           f'-w {OUT}/wheels llama-cpp-python=={LLAMA_CPP_PYTHON}', log=f"{OUT}/logs/build_llama_cpp_python.log")
        wheels = glob.glob(f"{OUT}/wheels/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
    sh(f"pip install -q {wheels[0]} lm_eval==0.4.13 datasets==5.1.0", log=f"{OUT}/logs/pip.log")
import importlib, llama_cpp, lm_eval, datasets
gpu_ok = llama_cpp.llama_supports_gpu_offload()
if IN_COLAB and not gpu_ok:
    raise RuntimeError("llama-cpp-python has no GPU offload: delete OUT/wheels and rerun to rebuild it with CUDA")
ENV["packages"] = {"llama_cpp_python": llama_cpp.__version__, "lm_eval": lm_eval.__version__,
                   "datasets": datasets.__version__, "gpu_offload": gpu_ok}
save_env(); print(ENV["packages"])

In [ ]:
# G9's runner (profile, task list, record format) from upstream main; its sha256 goes into every summary
sh(f"curl -fsSL {G9_RAW} -o {CODE}/run_eval_mini.py")
sys.path.insert(0, CODE)

In [ ]:
# bf16 GGUF: the llama.cpp converter at the team's build, pinned checkpoint; llama-quantize built CPU-only (~5 min)
NAME = MODEL.split("/")[-1]
BF16 = f"{WORK}/{NAME}-bf16.gguf"
QUANTIZE = shutil.which("llama-quantize") if LOCAL_TEST else f"{WORK}/llama.cpp/build/bin/llama-quantize"
if not LOCAL_TEST and not os.path.exists(QUANTIZE):
    if not os.path.exists(f"{WORK}/llama.cpp"):
        sh(f"git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp")
    sh(f"cmake -S {WORK}/llama.cpp -B {WORK}/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF "
       f"&& cmake --build {WORK}/llama.cpp/build -j --target llama-quantize", log=f"{OUT}/logs/build_llama_quantize.log")
if not os.path.exists(BF16):
    sh(f"pip install -q {WORK}/llama.cpp/gguf-py sentencepiece", log=f"{OUT}/logs/pip.log")
    from huggingface_hub import snapshot_download
    hf = snapshot_download(MODEL, revision=REVISION, local_dir=f"{WORK}/hf",
                           allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    sh(f"python {WORK}/llama.cpp/convert_hf_to_gguf.py {hf} --outtype bf16 --outfile {BF16}", log=f"{OUT}/logs/convert.log")
    shutil.rmtree(f"{WORK}/hf", ignore_errors=True)
ENV["bf16_gguf_bytes"] = os.path.getsize(BF16); save_env()
print(BF16, round(os.path.getsize(BF16) / 1e9, 2), "GB")

In [ ]:
# every format: quantize (same rule as scripts/quantize_eval.py: output layer Q8_0 below 8 bits), run G9's profile,
# save after each task. A format whose five tasks are all in its summary.json is skipped.
TASKS = ["arc_challenge", "hellaswag", "mmlu", "gsm8k", "truthfulqa_mc2"]
RUNS = f"{OUT}/runs" + (f"_limit{LIMIT}" if LIMIT else "")     # test runs never mix with the full run

def finished(fmt):
    p = f"{RUNS}/{fmt}/summary.json"
    return os.path.exists(p) and {r["task"] for r in json.load(open(p))["records"]} >= set(TASKS)

def run_format(fmt, gguf, gpu_layers):
    sh(f"cd {CODE} && {sys.executable} run_mini_gguf.py --gguf {gguf} --format {fmt} --out {RUNS} "
       f"--g9-harness {CODE}/run_eval_mini.py --n-gpu-layers {gpu_layers}" + (f" --limit {LIMIT}" if LIMIT else ""),
       log=f"{OUT}/logs/run_{fmt}.log")

for fmt in FORMATS:
    if finished(fmt):
        print(f"{fmt}: all tasks done, skipped"); continue
    t0 = time.time()
    gguf = BF16 if fmt == "bf16" else f"{WORK}/{NAME}-{fmt}.gguf"
    made_here = False
    if not os.path.exists(gguf):
        extra = "" if fmt == "Q8_0" else "--output-tensor-type q8_0"
        sh(f"{QUANTIZE} {extra} {BF16} {gguf} {fmt}", log=f"{OUT}/logs/quantize_{fmt}.log"); made_here = True
    gpu_layers = -1
    try:
        run_format(fmt, gguf, gpu_layers)
    except RuntimeError:
        if fmt != "bf16":
            raise
        gpu_layers = BF16_FALLBACK_GPU_LAYERS   # bf16 did not fit with every layer on the GPU
        print(f"bf16 failed with all layers on the GPU; retrying with {gpu_layers} GPU layers")
        run_format(fmt, gguf, gpu_layers)
    ENV["formats"][fmt] = {"gpu_layers": gpu_layers, "gguf_bytes": os.path.getsize(gguf),
                           "wall_clock_s": round(time.time() - t0), "finished": datetime.datetime.now().isoformat(timespec="seconds")}
    save_env()
    if made_here and fmt != "bf16" and not LOCAL_TEST:
        os.remove(gguf)   # keep the local disk small

In [ ]:
# results: G9-primary scores per format, the paired comparison with the base, a zip of everything
import pandas as pd
rows = []
for p in sorted(glob.glob(f"{RUNS}/*/summary.json")):
    s = json.load(open(p))
    for r in s["records"]:
        if r["primary"]:
            rows.append({"format": s["run_meta"]["format"], "task": r["task"], "points": round(100 * r["value"], 1),
                         "stderr": round(100 * (r["stderr"] or 0), 1), "n": r["n_items"], "seconds": r["wall_clock_s"]})
long = pd.DataFrame(rows)
long.to_csv(f"{RUNS}/scores_long.csv", index=False)
scores = long.pivot_table(index="format", columns="task", values="points").reindex([f for f in FORMATS if f in set(long.format)])
scores.to_csv(f"{RUNS}/scores.csv")
print(scores.to_string())
sh(f"{sys.executable} {CODE}/paired.py {RUNS} --base {BASE} --out {RUNS}/paired.md")
ENV["finished"] = datetime.datetime.now().isoformat(timespec="seconds"); save_env()
shutil.copy(f"{OUT}/environment.json", f"{RUNS}/environment.json")
shutil.copytree(f"{OUT}/logs", f"{RUNS}/logs", dirs_exist_ok=True)
zip_path = shutil.make_archive(RUNS, "zip", RUNS)
print("\nEverything is in", RUNS, "and", zip_path)
n_files = sum(len(f) for _, _, f in os.walk(RUNS)); print(n_files, "files:", sorted(os.listdir(RUNS)))

In [ ]:
# bring the results back to the machine running VS Code: the zip, as base64 text in this cell's output. With a
# Colab kernel every file above lives on Google's VM; cell outputs are what VS Code keeps when you save the
# notebook. Locally: python extract_results.py eval_mini_colab.ipynb  -> writes the zip next to the notebook.
import base64
blob = base64.b64encode(open(zip_path, "rb").read()).decode()
print(f"EVAL_MINI_ZIP_BEGIN {os.path.basename(zip_path)} {len(blob)}")
for i in range(0, len(blob), 4000):
    print(blob[i:i + 4000])
print("EVAL_MINI_ZIP_END")

## Send back

**Save the notebook in VS Code** (the last cell's output carries the results), then locally run
`python extract_results.py eval_mini_colab.ipynb`. It writes the zip (`runs_limit5.zip` for the test run, `runs.zip`
for the full run): per format a `summary.json`
(G9's record format) and one `*.samples.jsonl` per task; `scores.csv`, `scores_long.csv`, `paired.md`,
`environment.json` (GPU, versions, settings, per-format GPU layers and time) and `logs/`. From Drive, or if Drive
did not mount, download it from the runtime before the session ends.